# Model training: fine-tune + proof

Fine-tunes a SentenceTransformer on `triplets.csv` (attach the scraping notebook's output dataset),
evaluates held-out triplet accuracy, and prints the proof table.
Needs GPU + Internet ON. Set `QUICK_MODE = True` for a 5-minute smoke test.


In [ ]:
!pip install -q sentence-transformers peft 2>&1 | tail -1
import torch
print('cuda:', torch.cuda.is_available())


In [ ]:
# Config — attach the scraping output dataset, then set its path here
TRIPLETS = '/kaggle/input/arxiv-triplets/triplets.csv'  # <-- your dataset slug
BASE = 'allenai/specter2_base'   # the model behind the TextTAGC-beating numbers
USE_LORA = True                 # LoRA adapters (~1% params), as in the local runs
EPOCHS, BS, LR, SEQ, LIMIT = 2, 16, 2e-5, 128, 8000
QUICK_MODE = False              # True -> MiniLM, 2000 triplets, 1 epoch (smoke test)
if QUICK_MODE:
    BASE, EPOCHS, BS, LIMIT = 'sentence-transformers/all-MiniLM-L6-v2', 1, 32, 2000
OUT = '/kaggle/working/model-out'
import os
assert os.path.exists(TRIPLETS), f'missing {TRIPLETS}: attach the triplets dataset first'
print('ok:', TRIPLETS)


In [ ]:
# Train/val split by anchor (no query leaks across) + fine-tune loop
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

df = pd.read_csv(TRIPLETS).dropna()
if len(df) > LIMIT:
    df = df.sample(n=LIMIT, random_state=42).reset_index(drop=True)
anchors = df['anchor_text'].drop_duplicates().tolist()
tr_a, te_a = train_test_split(anchors, test_size=0.2, random_state=42)
tr = df[df['anchor_text'].isin(set(tr_a))]
te = df[df['anchor_text'].isin(set(te_a))]
print(f'triplets={len(df)} train={len(tr)} heldout={len(te)}')
mk = lambda d: [InputExample(texts=[r.anchor_text, r.positive_text, r.negative_text]) for r in d.itertuples()]

model = SentenceTransformer(BASE)
model.max_seq_length = SEQ
if USE_LORA and 'MiniLM' not in BASE:
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                     target_modules=['query', 'value', 'key', 'dense'])
    model[0].model = get_peft_model(model[0].auto_model, cfg)
    print('LoRA on')
loss = losses.MultipleNegativesRankingLoss(model=model, scale=20.0)
model.fit(train_objectives=[(DataLoader(mk(tr), batch_size=BS, shuffle=True), loss)],
          epochs=EPOCHS, warmup_steps=50, optimizer_params={'lr': LR}, show_progress_bar=True)
model.save(OUT)
print('saved:', OUT)
te.to_csv('/kaggle/working/heldout.csv', index=False)


In [ ]:
# Proof: held-out triplet accuracy (cos(anchor,pos) > cos(anchor,neg))
import torch, numpy as np
m = SentenceTransformer(OUT)
m.max_seq_length = SEQ
A = m.encode(te['anchor_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
P = m.encode(te['positive_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
N = m.encode(te['negative_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
A /= (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
P /= (np.linalg.norm(P, axis=1, keepdims=True) + 1e-12)
N /= (np.linalg.norm(N, axis=1, keepdims=True) + 1e-12)
acc = float(((A * P).sum(1) > (A * N).sum(1)).mean())
print(f'held-out triplet accuracy = {acc:.4f} (n={len(te)}, random = 0.50)')
print('paper context: TextTAGC AAN MRR 0.42978 | SciNUP kNN-SciBERT MAP 0.0232/MRR 0.2182')
print('local precedent, same recipe: specter2_ft MRR 0.4538 / fusion 0.4721 (SciNUP-100)')


## Retrieval metrics vs the papers (MRR / MAP / Recall)
Scores the trained model on SciNUP (1000 authors x 1000 candidates),
fuses with BM25, and prints the comparison table.


In [ ]:
# Eval data: SciNUP profiles + candidates + ground-truth qrels
import os
os.makedirs('/kaggle/working/scinup', exist_ok=True)
!cd /kaggle/working/scinup && [ -f dataset.jsonl ] || (wget -q https://iai.group/downloads/SciNUP/dataset.tgz && tar xzf dataset.tgz && rm dataset.tgz sampled_users.tgz)
!cd /kaggle/working/scinup && [ -f qrels.trec ] || wget -q -O qrels.trec https://raw.githubusercontent.com/iai-group/SciNUP/main/data/retrieval_results/ground_truth_qrels.trec
!ls -la /kaggle/working/scinup/


In [ ]:
import json
import numpy as np
from collections import defaultdict

def doc_text(c):
    return f"Title: {c['title'].strip()} Abstract: {c['abstract'].strip()}"

def load_records(path, limit=0):
    recs = [json.loads(l) for l in open(path, encoding='utf-8')]
    return recs[:limit] if limit else recs

def dense_run(records, model, top_k=100, bs=64):
    runs = {}
    for i, rec in enumerate(records):
        cands = rec['candidate_items']
        ids = [c['article_id'] for c in cands]
        V = model.encode([doc_text(c) for c in cands], batch_size=bs,
                         normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        q = model.encode([rec['nl_profile']], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False)[0]
        runs[rec['author_id']] = list(zip(ids, map(float, V @ q)))
        if (i + 1) % 100 == 0:
            print(f"dense {i + 1}/{len(records)}", flush=True)
    return runs

def bm25_run(records):
    from rank_bm25 import BM25Okapi
    runs = {}
    for i, rec in enumerate(records):
        ids = [c['article_id'] for c in rec['candidate_items']]
        corpus = [doc_text(c).lower().split() for c in rec['candidate_items']]
        s = BM25Okapi(corpus).get_scores(rec['nl_profile'].lower().split())
        runs[rec['author_id']] = list(zip(ids, map(float, s)))
        if (i + 1) % 200 == 0:
            print(f"bm25 {i + 1}/{len(records)}", flush=True)
    return runs

def rrf(lists, k=60):
    out = {}
    for aid in lists[0]:
        sc = defaultdict(float)
        for runs in lists:
            items = runs.get(aid, []); items = sorted(items.items(), key=lambda x: x[1], reverse=True) if isinstance(items, dict) else sorted(items, key=lambda x: x[1], reverse=True);
            for r, (doc, _) in enumerate(items, 1):
                sc[doc] += 1.0 / (k + r)
        out[aid] = sorted(sc.items(), key=lambda x: x[1], reverse=True)
    return out

def read_trec(path, qrel):
    d = defaultdict(dict)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) < 4:
            continue
        d[p[0]][p[2]] = float(p[3]) if qrel else float(p[4])
    return d

def trec_metrics(run, qrels, k=100):
    R = M = RR = ND = n = 0.0
    cnt = 0
    for qid, gold in qrels.items():
        pos = {d for d, r in gold.items() if r > 0}
        if not pos:
            continue
        cnt += 1
        order = [d for d, _ in sorted(run.get(qid, {}).items(), key=lambda x: x[1], reverse=True)]
        hits = ap = first = 0
        for i, d in enumerate(order, 1):
            if d in pos:
                hits += 1
                if not first:
                    first = i
                ap += hits / i
        M += ap / len(pos)
        RR += 1.0 / first if first else 0.0
        R += sum(1 for d in order[:k] if d in pos) / len(pos)
        dcg = sum(1.0 / (2 ** r - 1) for r, d in enumerate(order[:10], 1) if d in pos)
        ideal = sum(1.0 / (2 ** r - 1) for r in range(1, min(10, len(pos)) + 1))
        ND += dcg / ideal if ideal else 0.0
    rec = R / cnt
    return {'n': cnt, 'R@100': rec, 'MAP': M / cnt, 'MRR': RR / cnt, 'NDCG@10': ND / cnt}

def write_trec(path, runs, name, top_k=100):
    with open(path, 'w') as f:
        for aid, items in runs.items():
            seen, rank = set(), 1
            for doc, s in sorted(items, key=lambda x: x[1], reverse=True)[:top_k]:
                if doc in seen:
                    continue
                seen.add(doc)
                f.write(f'{aid} Q0 {doc} {rank} {s:.4f} {name}' + chr(10))
                rank += 1
print('eval helpers ok')


In [ ]:
from sentence_transformers import SentenceTransformer
records = load_records('/kaggle/working/scinup/dataset.jsonl')
print('authors:', len(records))
ft = SentenceTransformer(OUT)
runs_ft = dense_run(records, ft, top_k=100, bs=64)
write_trec('/kaggle/working/bge_ft.trec', runs_ft, 'bge_ft')
print('wrote bge_ft.trec')


In [ ]:
runs_bm25 = bm25_run(records)
runs_rrf = rrf([runs_ft, runs_bm25])
qrels = read_trec('/kaggle/working/scinup/qrels.trec', True)
print(f"{'run':<10}{'R@100':>8}{'MAP':>8}{'MRR':>8}{'NDCG@10':>9}")
for name, r in [('ft', runs_ft), ('bm25', runs_bm25), ('rrf', runs_rrf)]:
    m = trec_metrics({a: dict(v) for a, v in r.items()}, qrels)
    print(f"{name:<10}{m['R@100']:>8.4f}{m['MAP']:>8.4f}{m['MRR']:>8.4f}{m['NDCG@10']:>9.4f}  (n={m['n']})")
print('paper: TextTAGC AAN MRR 0.42978 | SciNUP kNN-SciBERT 0.1480/0.0232/0.2182/0.1019 | ensemble 0.4136/0.2163/0.6333/0.4481')


## 80/20 split diagnostics: ROC-AUC + confusion matrix
Uses the held-out triplets: every anchor–positive is label 1, every anchor–negative is label 0, scored by cosine similarity. No extra data needed.


In [ ]:
# ROC / AUC / confusion matrix on the held-out split (explicit 80-20 by anchor)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

full = pd.read_csv(TRIPLETS).dropna()
anchors80, anchors20 = train_test_split(full['anchor_text'].drop_duplicates().tolist(),
                                        test_size=0.2, random_state=42)
dia = full[full['anchor_text'].isin(set(anchors20))].reset_index(drop=True)
print(f'diagnostic split: train-anchors={len(anchors80)} test-anchors={len(anchors20)} pairs={len(dia)}')

E = m.encode
A = E(dia['anchor_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
P = E(dia['positive_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
N = E(dia['negative_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
for V in (A, P, N):
    V /= (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
pos_scores = (A * P).sum(1)
neg_scores = (A * N).sum(1)
y_true = np.array([1] * len(dia) + [0] * len(dia))
y_score = np.concatenate([pos_scores, neg_scores])
print(f'ROC-AUC = {roc_auc_score(y_true, y_score):.4f}')

# confusion matrix at threshold 0 (cosine sign) + best (Youden) threshold
fpr, tpr, thrs = roc_curve(y_true, y_score)
best = thrs[int(np.argmax(tpr - fpr))]
print(f"threshold cos>0:      {confusion_matrix(y_true, y_score > 0.0).ravel()}  # [TN FP FN TP]")
print(f"threshold youden={best:.3f}: {confusion_matrix(y_true, y_score > best).ravel()}  # [TN FP FN TP]")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr, tpr, label='model')
ax[0].plot([0, 1], [0, 1], '--', label='chance')
ax[0].set(xlabel='FPR', ylabel='TPR', title='ROC')
ax[0].legend()
cm = confusion_matrix(y_true, y_score > best)
im = ax[1].imshow(cm)
ax[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=['pred 0', 'pred 1'],
          yticklabels=['true 0', 'true 1'], title=f'Confusion @ {best:.3f}')
fig.colorbar(im, ax=ax[1])
plt.tight_layout()
plt.savefig('/kaggle/working/roc_cm.png' if os.path.exists('/kaggle/working') else 'roc_cm.png')
plt.show()
